# Notebook 06 — Process-Aware Utility Optimization

## Goal

Notebook05 showed that process-aware scoring can bring rescue clones into the Top-10 candidate pool.

However, Notebook05 did not yet prove that those rescue clones improve true selection quality.

The goal of Notebook06 is therefore:

> Can process-aware scoring improve the actual utility of selected clones?

---

## Key difference from Notebook05

Notebook05 asked:

> Can rescue clones enter the selection pool?

Notebook06 asks:

> Do process-aware selected clones better match the true best clones?

---

## Core idea

Instead of simply giving rescue clones a bonus, this notebook estimates expected post-optimization performance:

- expected productivity after process optimization
- expected stability after process optimization
- expected aggregation after process optimization

Then clones are ranked using this expected optimized profile.

---

## Important limitation

This notebook is still a simulation.

The process effect is not learned from real media/feed/process data yet.  
It is a structured hypothesis layer that will later be replaced by a real clone × process model.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

scenario = "legacy"
n_clones = 5000

PRED_PATH = Path("../data/synthetic/processed") / f"predictions_03b_qp_{n_clones}_{scenario}.csv"
df = pd.read_csv(PRED_PATH)

df.head()

,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob,rescue_upside_qp,rescue_stability_band,rescue_quality_band,rescue_aggressive_penalty,rescue_not_already_pass,pred_rescue_score,pred_rescue_label
0,CLONE_1502,0.498165,2.673129e-08,8.435996,0.238692,0,0.206868,3.215605e-08,4.404966,1,0.001494,0.003219,0.000312,0.506117,0.981713,0.996769,0.710487,0.568590,0
1,CLONE_2587,0.373417,8.407121e-08,3.130765,0.464397,0,0.600740,5.393417e-08,3.662670,0,0.007656,0.000000,0.013124,0.921942,0.000000,1.000000,0.435364,0.416664,0
2,CLONE_2654,0.461606,2.770586e-08,7.582349,0.256485,0,0.156409,3.459109e-08,10.144181,1,0.001523,0.004162,0.000529,0.627981,0.737814,0.995823,0.688798,0.538817,0
3,CLONE_1056,0.348761,7.746555e-08,6.885601,0.507813,1,0.465880,6.392818e-08,2.383546,0,0.012355,0.000000,0.011648,0.991742,0.538743,1.000000,0.382442,0.594558,0
4,CLONE_0706,0.644735,1.744327e-07,8.275030,0.018172,0,0.678633,1.526521e-07,5.032915,0,0.004539,0.039329,0.033316,0.017551,0.935723,0.960529,0.979289,0.407241,0


## Step 1 — Define baseline utility

We first define the baseline utility score.

This utility represents the expected value of each clone before any process optimization is applied.

For biosimilar-style selection, the utility prioritizes:

- high late qP
- low qP drop
- low aggregation, but with a smaller penalty than productivity and stability

The true utility is used only for offline evaluation because true late-stage outcomes would not be available during real clone selection.

In [2]:
# --------------------------------------------------
# Step 1 — Baseline utility definition
# --------------------------------------------------
# Assumption:
# df is already loaded from predictions_03b_qp_*.csv
#
# Required columns:
# - pred_late_qp
# - pred_qp_drop
# - pred_late_agg
# - true_late_qp
# - true_qp_drop
# - true_late_agg
# - pred_rescue_score
# - pred_rescue_label

def z(s):
    """
    Standardize a numeric series to z-score.
    
    Higher z-score means higher relative value.
    For negative objectives such as qP drop or aggregation,
    we subtract the z-score later.
    """
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)


def z01(s):
    """
    Normalize a numeric series to [0, 1].
    
    This is useful for constructing process sensitivity scores
    where we want all components to contribute on a comparable scale.
    """
    s = pd.Series(s).astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)


def make_true_utility(df):
    """
    Offline evaluation utility.
    
    This represents the true quality of a clone after late-stage outcomes
    are already known. It is used only for validation.
    """
    return (
        1.0 * z(df["true_late_qp"])
        - 1.0 * z(df["true_qp_drop"])
        - 0.2 * z(df["true_late_agg"])
    )


def make_pred_utility(df):
    """
    Baseline predicted utility.
    
    This represents how clones would be ranked using predicted late-stage
    outcomes only, before any process-aware correction.
    """
    return (
        1.0 * z(df["pred_late_qp"])
        - 1.0 * z(df["pred_qp_drop"])
        - 0.2 * z(df["pred_late_agg"])
    )


# Create baseline scores
df["true_util"] = make_true_utility(df)
df["baseline_score"] = make_pred_utility(df)

display(df[[
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_rescue_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
    "baseline_score",
]].head())

,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_util,baseline_score
0,CLONE_1502,2.673129e-08,0.498165,8.435996,0.568590,3.215605e-08,0.206868,4.404966,1.550478,-0.977650
1,CLONE_2587,8.407121e-08,0.373417,3.130765,0.416664,5.393417e-08,0.600740,3.662670,-0.804406,0.773624
2,CLONE_2654,2.770586e-08,0.461606,7.582349,0.538817,3.459109e-08,0.156409,10.144181,1.502466,-0.568919
3,CLONE_1056,7.746555e-08,0.348761,6.885601,0.594558,6.392818e-08,0.465880,2.383546,0.098570,0.649212
4,CLONE_0706,1.744327e-07,0.644735,8.275030,0.407241,1.526521e-07,0.678633,5.032915,-1.355847,-1.890491


## Step 2 — Estimate process sensitivity

This step creates a simple process sensitivity score.

The biological idea is:

> Some clones may not look ideal under the baseline condition, but may respond better to process optimization.

We approximate this sensitivity using three signals:

1. `pred_rescue_score`
   - higher means the clone was already identified as a process-optimization candidate

2. `pred_qp_drop`
   - moderate instability may indicate room for improvement through process control

3. `pred_late_agg`
   - aggregation risk may be partially improved by process or media optimization

This is not yet a real process model.  
It is a structured proxy for potential process responsiveness.

In [3]:
# --------------------------------------------------
# Step 2 — Process sensitivity model
# --------------------------------------------------
# The score estimates how much a clone may benefit from process optimization.
#
# Higher process_sensitivity means:
# - more room for productivity improvement
# - more room for stability improvement
# - more room for aggregation reduction

# Ensure rescue columns exist
if "pred_rescue_score" not in df.columns:
    df["pred_rescue_score"] = 0.0

if "pred_rescue_label" not in df.columns:
    df["pred_rescue_label"] = 0

# Process sensitivity components
df["sens_rescue"] = z01(df["pred_rescue_score"])
df["sens_instability"] = z01(df["pred_qp_drop"])
df["sens_aggregation"] = z01(df["pred_late_agg"])

# Combined process sensitivity
df["process_sensitivity"] = (
    0.50 * df["sens_rescue"]
    + 0.30 * df["sens_instability"]
    + 0.20 * df["sens_aggregation"]
)

print("=== Process sensitivity summary ===")
display(df["process_sensitivity"].describe())

display(df[[
    "clone_id",
    "pred_rescue_score",
    "pred_qp_drop",
    "pred_late_agg",
    "sens_rescue",
    "sens_instability",
    "sens_aggregation",
    "process_sensitivity",
]].sort_values("process_sensitivity", ascending=False).head(15))

=== Process sensitivity summary ===


count    1000.000000
mean        0.398586
std         0.095481
min         0.153854
25%         0.313200
50%         0.380077
75%         0.500166
max         0.689495
Name: process_sensitivity, dtype: float64

,clone_id,pred_rescue_score,pred_qp_drop,pred_late_agg,sens_rescue,sens_instability,sens_aggregation,process_sensitivity
524,CLONE_2171,0.711942,0.713485,8.462717,0.711942,0.708773,0.604461,0.689495
180,CLONE_4625,1.000000,0.322967,8.670837,1.000000,0.128861,0.623753,0.663409
505,CLONE_0048,0.701157,0.564631,10.686979,0.701157,0.487727,0.810638,0.659024
730,CLONE_4878,0.731989,0.545547,9.662363,0.731989,0.459389,0.715662,0.646943
63,CLONE_3863,0.581613,0.738468,7.816017,0.581613,0.745871,0.544516,0.623471
786,CLONE_3928,0.422060,0.782572,9.758332,0.422060,0.811366,0.724558,0.599351
965,CLONE_4665,0.353965,0.783414,10.712078,0.353965,0.812616,0.812965,0.583360
666,CLONE_0535,0.627884,0.507735,9.861568,0.627884,0.403238,0.734127,0.581739
119,CLONE_2352,0.454161,0.725135,8.502395,0.454161,0.726072,0.608139,0.566530
565,CLONE_4863,0.295950,0.893325,8.474223,0.295950,0.975831,0.605528,0.561830


## Step 3 — Simulate expected process gains

Here we convert process sensitivity into expected performance changes.

We simulate three effects:

1. Productivity gain
   - process optimization may increase late qP

2. Stability gain
   - process optimization may reduce qP drop

3. Quality gain
   - process optimization may reduce aggregation

This is the first key difference from Notebook05:

Notebook05 used process-aware scoring mainly as a ranking boost.  
Notebook06 converts process potential into expected biological performance changes.

In [4]:
# --------------------------------------------------
# Step 3 — Expected process gain model
# --------------------------------------------------
# These coefficients represent assumed strength of process optimization.
#
# They are intentionally moderate.
# If the effect is too strong, the model will unrealistically promote rescue clones.
# If too weak, optimization will not change ranking.

ALPHA_QP = 0.25       # expected relative productivity improvement
BETA_DROP = 0.15      # expected absolute qP drop reduction
GAMMA_AGG = 0.25      # expected relative aggregation reduction

# Expected improvements
df["expected_qp_gain_frac"] = ALPHA_QP * df["process_sensitivity"]
df["expected_drop_reduction"] = BETA_DROP * df["process_sensitivity"]
df["expected_agg_reduction_frac"] = GAMMA_AGG * df["process_sensitivity"]

# Apply expected process effects
df["opt_late_qp"] = df["pred_late_qp"] * (1.0 + df["expected_qp_gain_frac"])

df["opt_qp_drop"] = (
    df["pred_qp_drop"] - df["expected_drop_reduction"]
).clip(lower=0.0)

df["opt_late_agg"] = (
    df["pred_late_agg"] * (1.0 - df["expected_agg_reduction_frac"])
).clip(lower=0.0)

print("=== Expected process gain summary ===")
display(df[[
    "expected_qp_gain_frac",
    "expected_drop_reduction",
    "expected_agg_reduction_frac",
]].describe())

display(df[[
    "clone_id",
    "process_sensitivity",
    "pred_late_qp",
    "opt_late_qp",
    "pred_qp_drop",
    "opt_qp_drop",
    "pred_late_agg",
    "opt_late_agg",
    "pred_rescue_score",
    "pred_rescue_label",
]].sort_values("process_sensitivity", ascending=False).head(15))

=== Expected process gain summary ===


,expected_qp_gain_frac,expected_drop_reduction,expected_agg_reduction_frac
count,1000.000000,1000.000000,1000.000000
mean,0.099647,0.059788,0.099647
std,0.023870,0.014322,0.023870
min,0.038464,0.023078,0.038464
25%,0.078300,0.046980,0.078300
50%,0.095019,0.057012,0.095019
75%,0.125042,0.075025,0.125042
max,0.172374,0.103424,0.172374


,clone_id,process_sensitivity,pred_late_qp,opt_late_qp,pred_qp_drop,opt_qp_drop,pred_late_agg,opt_late_agg,pred_rescue_score,pred_rescue_label
524,CLONE_2171,0.689495,4.044897e-06,4.742132e-06,0.713485,0.610061,8.462717,7.003966,0.711942,1
180,CLONE_4625,0.663409,4.500575e-06,5.247005e-06,0.322967,0.223455,8.670837,7.232759,1.000000,1
505,CLONE_0048,0.659024,4.109802e-06,4.786917e-06,0.564631,0.465777,10.686979,8.926234,0.701157,1
730,CLONE_4878,0.646943,3.511339e-06,4.079248e-06,0.545547,0.448506,9.662363,8.099613,0.731989,1
63,CLONE_3863,0.623471,3.052754e-06,3.528580e-06,0.738468,0.644947,7.816017,6.597751,0.581613,0
786,CLONE_3928,0.599351,1.308706e-06,1.504800e-06,0.782572,0.692670,9.758332,8.296165,0.422060,0
965,CLONE_4665,0.583360,1.162539e-06,1.332083e-06,0.783414,0.695910,10.712078,9.149828,0.353965,0
666,CLONE_0535,0.581739,1.638482e-06,1.876775e-06,0.507735,0.420474,9.861568,8.427353,0.627884,0
119,CLONE_2352,0.566530,7.487332e-07,8.547782e-07,0.725135,0.640155,8.502395,7.298179,0.454161,0
565,CLONE_4863,0.561830,3.448040e-08,3.932343e-08,0.893325,0.809050,8.474223,7.283955,0.295950,0


## Step 4 — Compute process-aware utility score

Now we compute the optimized utility score using expected post-optimization values.

The optimized score uses:

- `opt_late_qp`
- `opt_qp_drop`
- `opt_late_agg`

This score answers:

> If this clone receives process optimization, how good do we expect it to become?

This is different from simply giving rescue clones a fixed bonus.

In [5]:
# --------------------------------------------------
# Step 4 — Process-aware utility score
# --------------------------------------------------

def make_process_aware_utility(df):
    """
    Expected post-optimization utility.
    
    Higher is better.
    """
    return (
        1.0 * z(df["opt_late_qp"])
        - 1.0 * z(df["opt_qp_drop"])
        - 0.2 * z(df["opt_late_agg"])
    )

df["process_aware_score"] = make_process_aware_utility(df)

display(df[[
    "clone_id",
    "baseline_score",
    "process_aware_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "pred_late_qp",
    "opt_late_qp",
    "pred_qp_drop",
    "opt_qp_drop",
    "pred_late_agg",
    "opt_late_agg",
]].sort_values("process_aware_score", ascending=False).head(15))

,clone_id,baseline_score,process_aware_score,process_sensitivity,pred_rescue_score,pred_rescue_label,pred_late_qp,opt_late_qp,pred_qp_drop,opt_qp_drop,pred_late_agg,opt_late_agg
180,CLONE_4625,12.718873,13.376981,0.663409,1.000000,1,4.500575e-06,5.247005e-06,0.322967,0.223455,8.670837,7.232759
621,CLONE_3254,12.974673,13.026362,0.516945,0.787107,1,4.328425e-06,4.887815e-06,0.246238,0.168696,8.356089,7.276179
505,CLONE_0048,9.293532,9.876784,0.659024,0.701157,1,4.109802e-06,4.786917e-06,0.564631,0.465777,10.686979,8.926234
524,CLONE_2171,7.964768,8.643666,0.689495,0.711942,1,4.044897e-06,4.742132e-06,0.713485,0.610061,8.462717,7.003966
730,CLONE_4878,7.933686,8.447124,0.646943,0.731989,1,3.511339e-06,4.079248e-06,0.545547,0.448506,9.662363,8.099613
986,CLONE_2776,7.992076,8.173363,0.560216,0.577185,0,3.750558e-06,4.275839e-06,0.636881,0.552849,6.964564,5.989149
314,CLONE_1619,8.901264,8.093612,0.205382,0.358295,0,2.630590e-06,2.765659e-06,0.236190,0.205383,3.356785,3.184429
63,CLONE_3863,5.105205,5.491319,0.623471,0.581613,0,3.052754e-06,3.528580e-06,0.738468,0.644947,7.816017,6.597751
23,CLONE_0643,4.466508,4.624839,0.535567,0.650158,1,1.947179e-06,2.207890e-06,0.482194,0.401859,7.384080,6.395412
666,CLONE_0535,3.180570,3.455474,0.581739,0.627884,0,1.638482e-06,1.876775e-06,0.507735,0.420474,9.861568,8.427353


## Step 5 — Compare baseline vs process-aware Top-K selection

We compare the Top-10 clones selected by:

1. baseline predicted utility
2. process-aware utility

This tells us whether process-aware scoring changes the candidate pool.

Important interpretation:

- High overlap means the optimized score is conservative
- Low overlap means the optimized score strongly changes decisions
- Moderate overlap is usually desirable

In [6]:
# --------------------------------------------------
# Step 5 — Top-K selection comparison
# --------------------------------------------------

TOP_K = 10

baseline_top = df.sort_values("baseline_score", ascending=False).head(TOP_K).copy()
process_top = df.sort_values("process_aware_score", ascending=False).head(TOP_K).copy()

baseline_ids = set(baseline_top["clone_id"])
process_ids = set(process_top["clone_id"])

selection_overlap = len(baseline_ids & process_ids)

print("=== Top-K selection comparison ===")
print("Top-K:", TOP_K)
print("Baseline rescue count:", int(baseline_top["pred_rescue_label"].sum()))
print("Process-aware rescue count:", int(process_top["pred_rescue_label"].sum()))
print(f"Baseline vs process-aware overlap: {selection_overlap}/{TOP_K}")

print("\n=== Baseline Top10 ===")
display(baseline_top[[
    "clone_id",
    "baseline_score",
    "process_aware_score",
    "pred_rescue_score",
    "pred_rescue_label",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

print("\n=== Process-aware Top10 ===")
display(process_top[[
    "clone_id",
    "baseline_score",
    "process_aware_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "opt_late_qp",
    "opt_qp_drop",
    "opt_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

=== Top-K selection comparison ===
Top-K: 10
Baseline rescue count: 6
Process-aware rescue count: 6
Baseline vs process-aware overlap: 9/10

=== Baseline Top10 ===


,clone_id,baseline_score,process_aware_score,pred_rescue_score,pred_rescue_label,pred_late_qp,pred_qp_drop,pred_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
621,CLONE_3254,12.974673,13.026362,0.787107,1,4.328425e-06,0.246238,8.356089,0.000007,0.089331,11.556387,2.451734
180,CLONE_4625,12.718873,13.376981,1.000000,1,4.500575e-06,0.322967,8.670837,0.000012,0.084353,10.114743,3.040391
505,CLONE_0048,9.293532,9.876784,0.701157,1,4.109802e-06,0.564631,10.686979,0.000016,0.304950,12.743312,1.893128
314,CLONE_1619,8.901264,8.093612,0.358295,0,2.630590e-06,0.236190,3.356785,0.000003,0.164530,5.259631,1.978856
986,CLONE_2776,7.992076,8.173363,0.577185,0,3.750558e-06,0.636881,6.964564,0.000356,0.000000,8.477892,34.138263
524,CLONE_2171,7.964768,8.643666,0.711942,1,4.044897e-06,0.713485,8.462717,0.000007,0.839031,6.662415,-1.862933
730,CLONE_4878,7.933686,8.447124,0.731989,1,3.511339e-06,0.545547,9.662363,0.000005,0.417981,7.149145,0.525635
63,CLONE_3863,5.105205,5.491319,0.581613,0,3.052754e-06,0.738468,7.816017,0.000002,0.801780,5.469702,-1.962704
23,CLONE_0643,4.466508,4.624839,0.650158,1,1.947179e-06,0.482194,7.384080,0.000002,0.518817,6.462675,-0.332074
476,CLONE_2759,3.183487,3.140981,0.481011,0,8.872743e-07,0.273353,10.806613,0.000001,0.220844,16.093701,0.834379



=== Process-aware Top10 ===


,clone_id,baseline_score,process_aware_score,process_sensitivity,pred_rescue_score,pred_rescue_label,opt_late_qp,opt_qp_drop,opt_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
180,CLONE_4625,12.718873,13.376981,0.663409,1.000000,1,0.000005,0.223455,7.232759,0.000012,0.084353,10.114743,3.040391
621,CLONE_3254,12.974673,13.026362,0.516945,0.787107,1,0.000005,0.168696,7.276179,0.000007,0.089331,11.556387,2.451734
505,CLONE_0048,9.293532,9.876784,0.659024,0.701157,1,0.000005,0.465777,8.926234,0.000016,0.304950,12.743312,1.893128
524,CLONE_2171,7.964768,8.643666,0.689495,0.711942,1,0.000005,0.610061,7.003966,0.000007,0.839031,6.662415,-1.862933
730,CLONE_4878,7.933686,8.447124,0.646943,0.731989,1,0.000004,0.448506,8.099613,0.000005,0.417981,7.149145,0.525635
986,CLONE_2776,7.992076,8.173363,0.560216,0.577185,0,0.000004,0.552849,5.989149,0.000356,0.000000,8.477892,34.138263
314,CLONE_1619,8.901264,8.093612,0.205382,0.358295,0,0.000003,0.205383,3.184429,0.000003,0.164530,5.259631,1.978856
63,CLONE_3863,5.105205,5.491319,0.623471,0.581613,0,0.000004,0.644947,6.597751,0.000002,0.801780,5.469702,-1.962704
23,CLONE_0643,4.466508,4.624839,0.535567,0.650158,1,0.000002,0.401859,6.395412,0.000002,0.518817,6.462675,-0.332074
666,CLONE_0535,3.180570,3.455474,0.581739,0.627884,0,0.000002,0.420474,8.427353,0.000002,0.512580,3.121897,-0.099830


## Step 6 — Utility overlap evaluation

This is the central validation metric.

Utility overlap compares:

1. the true Top-10 clones by `true_util`
2. the selected Top-10 clones by model score

For baseline:

> true Top-10 vs baseline predicted Top-10

For process-aware:

> true Top-10 vs process-aware Top-10

If process-aware overlap is higher than baseline overlap, then the process-aware scoring improved selection quality.

If overlap is unchanged, the process-aware model changes ranking but does not improve true selection.

If overlap decreases, the process model is adding noise or bias.

In [7]:
# --------------------------------------------------
# Step 6 — Utility overlap evaluation
# --------------------------------------------------

def topk_overlap(true_scores, pred_scores, k):
    """
    Compare the clone IDs selected by true utility vs predicted score.
    
    Returns:
    fraction of Top-K predicted selections that overlap with Top-K true selections.
    """
    true_top_idx = set(pd.Series(true_scores).nlargest(k).index)
    pred_top_idx = set(pd.Series(pred_scores).nlargest(k).index)
    return len(true_top_idx & pred_top_idx) / k

baseline_overlap = topk_overlap(df["true_util"], df["baseline_score"], TOP_K)
process_overlap = topk_overlap(df["true_util"], df["process_aware_score"], TOP_K)

print("=== Utility overlap ===")
print(f"Baseline utility overlap:      {baseline_overlap:.3f}")
print(f"Process-aware utility overlap: {process_overlap:.3f}")

if process_overlap > baseline_overlap:
    print("\nInterpretation: Process-aware scoring improved true Top-K recovery.")
elif process_overlap == baseline_overlap:
    print("\nInterpretation: Process-aware scoring changed or preserved ranking but did not improve true Top-K recovery.")
else:
    print("\nInterpretation: Process-aware scoring reduced true Top-K recovery; process assumptions may be too aggressive or misaligned.")

=== Utility overlap ===
Baseline utility overlap:      0.300
Process-aware utility overlap: 0.300

Interpretation: Process-aware scoring changed or preserved ranking but did not improve true Top-K recovery.


## Step 7 — True performance comparison

Utility overlap is strict because it only checks exact Top-10 set overlap.

Here we also compare the average true outcomes of the selected clones.

This helps answer:

> Even if exact Top-10 overlap does not improve, did the selected clones become biologically better on average?

We compare:

- true late qP
- true qP drop
- true late aggregation
- true utility

In [8]:
# --------------------------------------------------
# Step 7 — True performance comparison
# --------------------------------------------------

summary = pd.DataFrame({
    "baseline_top10": baseline_top[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
    "process_aware_top10": process_top[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
})

summary["direction"] = [
    "higher is better",
    "lower is better",
    "lower is better",
    "higher is better",
]

display(summary)

print("=== Interpretation guide ===")
print("- If true_late_qp increases: productivity improved")
print("- If true_qp_drop decreases: stability improved")
print("- If true_late_agg decreases: quality improved")
print("- If true_util increases: overall selection quality improved")

,baseline_top10,process_aware_top10,direction
true_late_qp,0.000041,0.000041,higher is better
true_qp_drop,0.344162,0.373335,lower is better
true_late_agg,8.998960,7.701780,lower is better
true_util,4.070468,3.977047,higher is better


=== Interpretation guide ===
- If true_late_qp increases: productivity improved
- If true_qp_drop decreases: stability improved
- If true_late_agg decreases: quality improved
- If true_util increases: overall selection quality improved


## Step 8 — Rescue-specific diagnostics

This step checks whether selected rescue clones are actually meaningful.

A good process-aware system should not simply select more rescue clones.  
It should select rescue clones with reasonable true performance.

We compare selected pass vs selected rescue clones inside the process-aware Top-10.

In [9]:
# --------------------------------------------------
# Step 8 — Rescue-specific diagnostics
# --------------------------------------------------

if process_top["pred_rescue_label"].sum() > 0:
    rescue_diag = process_top.groupby("pred_rescue_label")[[
        "process_sensitivity",
        "pred_rescue_score",
        "pred_late_qp",
        "opt_late_qp",
        "pred_qp_drop",
        "opt_qp_drop",
        "pred_late_agg",
        "opt_late_agg",
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean()

    display(rescue_diag)

    print("Note:")
    print("pred_rescue_label = 0 means selected non-rescue clones")
    print("pred_rescue_label = 1 means selected rescue clones")
else:
    print("No rescue clones selected in process-aware Top10.")

,process_sensitivity,pred_rescue_score,pred_late_qp,opt_late_qp,pred_qp_drop,opt_qp_drop,pred_late_agg,opt_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
pred_rescue_label,,,,,,,,,,,,
0,0.492702,0.536245,0.000003,0.000003,0.529818,0.455913,6.999733,6.049671,0.000090,0.369722,5.582281,8.513646
1,0.618564,0.763725,0.000004,0.000004,0.479177,0.386392,8.870511,7.489027,0.000008,0.375744,9.114779,0.952647


Note:
pred_rescue_label = 0 means selected non-rescue clones
pred_rescue_label = 1 means selected rescue clones


## Final Interpretation

Notebook06 tests whether process-aware scoring improves actual clone selection quality.

### What to check

1. Process-aware rescue count
   - Did rescue clones enter the Top-10?

2. Baseline vs process-aware overlap
   - Did the candidate pool change moderately?

3. Utility overlap
   - Did process-aware scoring recover more true Top-10 clones?

4. True performance comparison
   - Did selected clones improve in true productivity, stability, or aggregation?

---

## Possible outcomes

### Case 1 — Rescue increases and utility overlap improves

This is the ideal result.

It means process-aware scoring is identifying rescue clones that are truly valuable.

---

### Case 2 — Rescue increases but utility overlap is unchanged

This means the scoring creates diversity, but does not yet improve selection quality.

This is still useful as an exploration strategy, but the process model needs refinement.

---

### Case 3 — Rescue increases but utility overlap decreases

This means the process assumptions are too aggressive or biologically misaligned.

The model is promoting rescue clones that do not actually improve true utility.

---

## Current interpretation

Notebook06 should be used to decide whether process-aware rescue logic should remain a scoring layer, become a separate exploration layer, or be deferred until real process/media data are available.

---

## Next step

Notebook07 should introduce explicit clone × process interaction:

- nutrient-limited vs nutrient-rich conditions
- media/feed sensitivity
- productivity/stability/quality trade-offs
- process-specific response profiles